# Monitor Dashboard — SPX tail hedge

Read-mostly view of the current book: load a portfolio and check its health, roll status, and triggers. No position construction here — use Hedge Design for that.

In [ ]:
"""Monitor Dashboard — SPX tail-hedge program."""
# Imports
import os

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from deltadewa.analysis import (
    DataQuality,
    HedgeTriggerThresholds,
    PortfolioAnalyzer,
    ScenarioGridCache,
    assess_market_environment,
    build_monetization_plan,
    classify_portfolio_shape,
    compute_crash_convexity,
    evaluate_hedge_triggers,
    get_volatility_stats,
)
from deltadewa.dashboard import (
    CarryDisplay,
    ChangeLogDisplay,
    PositionAgingDisplay,
    PositionDetailDisplay,
    RollStatusDisplay,
    StressDashboard,
    render_crash_table,
    start_session,
)
from deltadewa.marketdata import MarketDataError
from deltadewa.reporting import (
    PortfolioChangeTracker,
    build_program_report,
    render_html,
)
from deltadewa.visualization import (
    plot_carry_vs_convexity,
    plot_crash_convexity,
    plot_greeks_consolidated,
)
from deltadewa.widgets import (
    HedgeHealthDashboard,
    NetHedgeSummary,
    PortfolioWidgets,
    build_env_gauges,
)


In [ ]:
# ── Live market data toggle ───────────────────────────────────────
# Set True for live CBOE/FRED data (requires internet:
# cdn.cboe.com, fred.stlouisfed.org). Automatically falls back to
# static if the network is unavailable. Default False = offline-safe.
# See /examples/.env.example for how to set USE_LIVE=True in your environment.
_USE_LIVE = os.environ.get("USE_LIVE", "False") == "True"

# Dashboard session: portfolio, market data provider, IPS policy,
# and GlobalAssumptions — all bootstrapped in one call.
# auto_load_default=False: Monitor starts empty. Load a portfolio
# explicitly via the import widget below.
ctx = start_session(
    role="monitor",
    globals_dict=globals(),
    auto_load_default=False,
    use_live_market_data=_USE_LIVE,
)

portfolio = ctx.portfolio
ips_config = ctx.ips_config
dashboard_config = ctx.dashboard_config
market_data = ctx.market_data
global_assumptions = ctx.global_assumptions
reporter = ctx.reporter
portfolio_changelog = ctx.changelog
portfolio_serializer = ctx.serializer
EXPORT_DIR = ctx.export_dir
today = ctx.today

portfolio_widgets = PortfolioWidgets(
    portfolio,
    portfolio_serializer,
    portfolio_changelog,
)

reporter.success("Setup complete.")

## Load portfolio

In [ ]:
# Import Widget and Portfolio Change Tracker

# Create tracker — seeds the baseline snapshot from the current portfolio state
position_tracker = PortfolioChangeTracker(
    portfolio=portfolio,
    logger=portfolio_changelog,
    reporter=reporter,
)

# Import widget — reset the tracker after a successful import so it
# doesn't diff against the pre-import snapshot
import_widget = portfolio_widgets.display_import(
    on_import_success=position_tracker.reset,
)
display(import_widget)

In [ ]:
# Shape guard — check portfolio structure once after load.
_shape = classify_portfolio_shape(portfolio)
_has_book = bool(portfolio.positions) or portfolio.underlying_quantity != 0
if _has_book and not _shape.is_conforming:
    display(HTML(
        '<div style="border-left:4px solid #b45309;'
        ' background:#fffbeb; padding:10px 14px;'
        ' border-radius:4px; max-width:800px;'
        ' font-family:sans-serif; font-size:13px;'
        ' color:#78350f;">'
        f'<b>⚠ Portfolio shape:</b> {_shape.notice}'
        '</div>',
    ))

In [ ]:
# Volatility stats. Note: setup_dashboard()/start_session()
# already synced portfolio.volatility to the average internally;
# this just reads the stats back out.
vol_stats = get_volatility_stats(portfolio)

### Market context

In [ ]:
display(global_assumptions.display())

try:
    current_spot = market_data.get_spot(portfolio.get_symbol())
except MarketDataError:
    current_spot = global_assumptions.spot_price.value

print(f"Spot: {current_spot:,.2f}")
print(f"VIX:  {market_data.get_vix():.2f}")
print(f"Data: {ctx.market_data_source}")

## Tier 1 — Core Hedge Metrics


### Net Hedge Summary

In [ ]:
# Hedge Summary
net_hedge_summary = NetHedgeSummary(portfolio)
display(net_hedge_summary.display())

### Hedge Health

In [ ]:
# Display Hedge Health Dashboard
health_dashboard = HedgeHealthDashboard(portfolio, config=dashboard_config)
dashboard_loader = health_dashboard.display_config_loader()
display(dashboard_loader)
display(health_dashboard.display())
# Update when portfolio changes
health_dashboard.update()

### Crash Payoff & Scenario Table

### Metric Definitions

**Crash payoff ratio** = gross hedge payoff in the −X% crash scenario /
premium paid. 8.5x means the hedge returns 8.5x its cost in that crash
(net-profit ratio = payoff ratio − 1). Based on premium paid where
available, else current mark (flagged).

**Convexity %** — net crash P&L (hedge gain minus the underlying book's
loss at the crash scenario) divided by book notional, as a signed percent.
Positive means the hedge more than covers the equity loss.
The IPS target band sets the acceptable range.

**Premium Basis** shown in chart annotations: `paid` = cost-basis from position entry prices; `mark (approx)` = today's mark (shown when any long put lacks an `entry_premium`).


In [ ]:
if not _shape.is_conforming:
    print(f"[shape] {_shape.notice}")
# Compute crash payoff result once — shared by table and chart below.
if portfolio.positions:
    _crash = compute_crash_convexity(
        portfolio,
        ips_convexity=(
            ctx.ips_config.convexity
            if ctx.ips_config is not None
            else None
        ),
    )
else:
    _crash = None

In [ ]:
# Crash Payoff & Scenario Table
if _crash is not None:
    render_crash_table(_crash)
else:
    print("No positions in portfolio yet.")

In [ ]:
# Crash Payoff Chart
if _crash is not None:
    plot_crash_convexity(_crash)
    plt.show()
    print(
        "Gross payoff ($) vs shock (%). Dashed line = premium paid; "
        "annotation = payoff ratio at IPS crash scenario.",
    )
else:
    print("No positions to chart.")

In [ ]:
# Carry vs Convexity — cost-vs-protection view
if _crash is not None:
    _carry_metrics = PortfolioAnalyzer(portfolio).calculate_carry_metrics()
    _ips_row = next(
        (
            r for r in _crash.scenario_rows
            if (
                _crash.ips_convexity is not None
                and r.shock_pct == _crash.ips_convexity.crash_scenario_pct
            )
        ),
        None,
    )
    display(plot_carry_vs_convexity(
        carry_cost=_carry_metrics["total_theta_annual"],
        convexity_pct=_ips_row.convexity_pct if _ips_row else None,
        ips_convexity=_crash.ips_convexity,
    ))
else:
    print("No positions to chart.")

### Cost of Carry

In [ ]:
# Theta Decay & Carry Analysis
carry_display = CarryDisplay(portfolio, reporter)
carry_display.display()

## Tier 2 — Market Environment


### Market environment

In [ ]:
# Market Environment — Tier-2 hedge-cost read
_env = assess_market_environment(
    market_data, dashboard_config=dashboard_config,
)
_is_static = _env.data_quality == DataQuality.STATIC
_unavailable = _env.data_quality == DataQuality.UNAVAILABLE

_VERDICT_COLOR = {
    "CHEAP": "#2e7d32",
    "EXPENSIVE": "#c62828",
    "FAIR": "#e65100",
}
_verdict_str = (
    str(_env.hedge_cost_verdict) if _env.hedge_cost_verdict else "—"
)
_verdict_color = _VERDICT_COLOR.get(_verdict_str, "#555555")


def _me_fmt(val: object, spec: str = ".1f") -> str:
    return f"{val:{spec}}" if val is not None else "—"


_vix_str = _me_fmt(_env.vix)
_regime_pct = _me_fmt(_env.regime_percentile, ".0f")
_regime_lbl = str(_env.regime_label) if _env.regime_label else "—"
_skew_idx = _me_fmt(_env.skew_index, ".1f")
_skew_pct = (
    f"{_env.skew_percentile * 100:.0f}th pct"
    if _env.skew_percentile is not None
    else "—"
)
_term = str(_env.term_shape) if _env.term_shape else "—"
_fwd = _me_fmt(_env.forward_vol_front_3m)

_warn_p = '<p style="margin:6px 0 0; color:#b45309; font-size:12px;">'
_warn_sfx = " — enable live market data for a real read</p>"
if _unavailable:
    _warn_html = _warn_p + "⚠ Market data unavailable" + _warn_sfx
elif ctx.market_data_source == "static (live unavailable)":
    _warn_html = (
        _warn_p
        + "⚠ Live data unreachable — showing static values</p>"
    )
elif _is_static:
    _warn_html = _warn_p + "⚠ Static/offline data" + _warn_sfx
else:
    _warn_html = ""

# Pre-build CSS and HTML spans so f-string template lines stay ≤ 80 chars
_cm = 'style="color:#64748b;"'
_vix_r = f"{_regime_lbl} ({_regime_pct}th pct)"
_vix_span = f"<span><b>VIX</b> {_vix_str} <span {_cm}>{_vix_r}</span></span>"
_skw = f"<span {_cm}>{_skew_pct}</span>"
_skew_span = f"<span><b>Skew</b> {_skew_idx} {_skw}</span>"
_s_hdr = "display:flex; justify-content:space-between; align-items:baseline;"
_s_a = "border-left:4px solid #3b82f6; background:#f0f6ff;"
_s_b = " padding:10px 14px; border-radius:4px; max-width:800px;"
_s_c = " font-family:monospace; font-size:13px; color:#1e293b;"
_s_outer = _s_a + _s_b + _s_c
_s_row_a = "margin-top:6px; display:flex; gap:20px;"
_s_row_b = " flex-wrap:wrap; color:#334155;"
_s_row = _s_row_a + _s_row_b

display(HTML(f"""
<div style="{_s_outer}">
  <div style="{_s_hdr}">
    <b style="font-size:14px; letter-spacing:.03em;">Market Environment</b>
    <b style="color:{_verdict_color}; font-size:14px;">{_verdict_str}</b>
  </div>
  <div style="{_s_row}">
    {_vix_span}
    {_skew_span}
    <span><b>Term</b> {_term}</span>
    <span><b>Fwd vol 1M→3M</b> {_fwd}</span>
  </div>
  {_warn_html}
</div>
"""))

In [ ]:
# Tier-2 environment gauges — renders pre-computed _env
display(build_env_gauges(_env))

## Tier 3 — Structural & Operational


### Consolidated Greeks

In [ ]:
# Consolidated Greeks Visualization
if len(portfolio.positions) > 0:
    fig = plot_greeks_consolidated(
        portfolio,
        top_n=5,
        figsize=(16, 20),
    )
    fig.subplots_adjust(hspace=0.75, wspace=0.2)
    plt.show()
else:
    print("No positions to analyze. Add positions in BUILD mode.")

### Hedge Decision Triggers

In [ ]:
# Hedge Descision Triggers
trigger_result = evaluate_hedge_triggers(
    portfolio,
    reporter,
    thresholds=(
        HedgeTriggerThresholds.from_ips(ips_config.triggers)
        if ips_config is not None
        else None
    ),
)

### Roll Status

In [ ]:
# Display Roll Status
if ips_config is not None:
    try:
        current_spot = market_data.get_spot(portfolio.get_symbol())
    except MarketDataError:
        current_spot = global_assumptions.spot_price.value
    RollStatusDisplay(portfolio, ips_config, reporter=reporter).display(
        current_spot,
    )

## Tier 4 — Tactical / Optional


### Delta Drift Detail

_Planned — requires net-delta series from position history. See hedge\_design workbench._


### Liquidity

_Planned (Phase D2) — requires a live options-chain feed for bid/ask spread and open-interest data._


## Part VII — Hedge Program Report

Assembles the pre-computed crash, carry, and market-environment results into a
single printable summary aligned with IPS Part VII.
Edit `_period_label` before running, then run the cell.

In [ ]:
_period_label = "Q2 2026"  # <- edit before running

if not portfolio.positions:
    print("Program Report: load a portfolio first.")
elif ips_config is None:
    print("Program Report: IPS config unavailable (check ips.yaml).")
else:
    _rpt_carry = (
        _carry_metrics
        if "_carry_metrics" in vars()
        else PortfolioAnalyzer(portfolio).calculate_carry_metrics()
    )
    _mon_plan = build_monetization_plan(
        portfolio, ips_config, market_env=_env,
    )
    _report = build_program_report(
        portfolio=portfolio,
        ips_config=ips_config,
        crash_result=_crash,
        carry_metrics=_rpt_carry,
        market_env=_env,
        period_label=_period_label,
        as_of=today,
        monetization_plan=_mon_plan,
    )
    _html = render_html(_report)
    display(HTML(_html))
    _report_path = EXPORT_DIR / f"program_report_{today}.html"
    _report_path.parent.mkdir(parents=True, exist_ok=True)
    _report_path.write_text(_html, encoding="utf-8")
    print(f"Saved: {_report_path}")

> **Phase-D follow-on:** this notebook can be run headless via
> [papermill](https://papermill.readthedocs.io/) by tagging `_period_label`
> (and optionally a `portfolio_path` parameter cell) — enabling scheduled,
> no-UI report generation. Not part of this branch.

## Positions & Reference


### Position Aging

In [ ]:
# Position Aging & Expiration Calendar
PositionAgingDisplay(portfolio, reporter).display()

### Position Detail

In [ ]:
# Position Detail Table
PositionDetailDisplay(portfolio).display()

### Current-book stress snapshot

In [ ]:
# Minimal stress setup for a single current-structure snapshot
scenario_cache = ScenarioGridCache(max_size=128)
analyzer = PortfolioAnalyzer(portfolio)
stress_dashboard = StressDashboard(
    portfolio=portfolio,
    analyzer=analyzer,
    cache=scenario_cache,
    global_assumptions=global_assumptions,
    reporter=reporter,
)

In [ ]:
# Interactive Stress Test Heatmap (Spot x Volatility)
if len(portfolio.positions) > 0:
    spot_vol_widget = stress_dashboard.create_spot_vol_heatmap(
        metric="pnl",
        days_forward=0,
    )
    display(spot_vol_widget)
else:
    reporter.error(
        "No positions to analyze. Add positions in BUILD mode first.",
    )

## Session


### Session Change Log

In [ ]:
# Portfolio Change Log Display
ChangeLogDisplay(portfolio_changelog, reporter).display()

### Export snapshot

In [ ]:
# Final Export Widget
final_export_widget = portfolio_widgets.display_export()
display(final_export_widget)

reporter.header("SESSION COMPLETE")
print("Remember to export your portfolio to save your work!")
print("Use the widget above to export in JSON, CSV, or YAML format.")
reporter.divider()